In [6]:
# -*- coding: utf-8 -*-
"""
Created on Wed Oct 21 14:12:42 2020

@author: clara, viggy
"""

import numpy as np
import matplotlib.pyplot as plt
import copy
import statistics
import pandas as pd
import os
import re
import glob
from scipy.signal import find_peaks
from func_stepFrequency import get_gait, get_peaks, calc_step_width, calculate_step_stride_length, calc_interstep_all, get_step_lengths, get_frames, variability, asymmetry, intersect, calculate_stance_swing, calculate_percent_cycle, double_support_time
import traceback

In [7]:
new_frames = pd.DataFrame()
error_subj = []
error_trace = []
error_idx = []

In [8]:
INDICES = ["trial_num", "test_type",
                      "Avg_LF_stance_time", 
                      "Avg_LF_swing_time",
                      "Avg_RF_stance_time",
                      "Avg_RF_swing_time",
                      "Variability_LF_stance_time",
                      "Variability_LF_swing_time",
                      "Variability_RF_stance_time",
                      "Variability_RF_swing_time",
                      "Asymmetry_Stance_time",
                      "Asymmetry_Swings_time",
                      "total_%_swing_time",
                      "total_%_stance_time",
                      "double_support_time",
                      "Avg_InterStepTime",
                      "Variability_InterStepTime",
                      "Frequency",
                      "Variance_COM_y",
                      "speed",
                      "Avg_stride_length",
                      "Variance_stride_length",
                      "Variability_stride_length",
                      "Asymmetry_stride_length",
                      "Avg_step_length",
                      "Variance_step_length",
                      "Variability_step_length",
                      "Avg_step_width",
                      "Variance_step_width",
                      "Variability_step_width",
                      "Avg_LF_max_height",
                      "Avg_RF_max_height",
                      "Variability_LF_max_height",
                      "Variability_RF_max_height",
                      "Asymmetry_F_max_height"]

## Load COM Data

In [9]:
#filelocation = glob.glob(f'../data/gang/*.txt')
#filelocation = glob.glob(f'Patient_txt_PD_reup/*.txt')
#filelocation = glob.glob(f'Patient_txt_ataxia_reup/*.txt')
#filelocation = glob.glob(f'missing_txt_files_20250818/*.txt')
filelocation = glob.glob(f'all_pat/*.txt')


indexes=['test_type','Step_Frequency', 'Av_Step_Length', 'Var_Step_Length',
         'Inter_Step_time', 'Var_InterStepTime', 'Speed', 'Var_COMy',
         'Step_width', 'LF_height', 'RF_height', 'Var_Step_width']
results = pd.DataFrame(index=indexes)

all_results = pd.DataFrame()
counter = 0
PATTERN = r'-[2-9]-'

for files in filelocation:
    try:
        data = np.loadtxt(files, encoding = 'latin1') 
        #sub_name = [s[0:6] for s in os.path.basename(files).split()]
        file_strip = os.path.basename(files)
        sub_name = file_strip[0:3]
        
        #search for multiple trials
        search = re.search(PATTERN, file_strip)
        if search != None: #match
            trial_num = search.group(0)[1]
        else: #no match
            trial_num = "1"
    
        subject = sub_name
        
        '''
        print("file_strip", file_strip)
        print("sub_name", sub_name)
        print("trial_num", trial_num)
        
        print(f'Starting analysis for subject: {subject}')
        print(f"---------index {counter}---------")
        '''
        
        if 'normalergang' in files.lower():
            test_type = 'NG'
            #sub_name += num
            #sub_name += "_NG"
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4', 'lane_5', 'lane_6']
        elif 'rückwärtsgang' in files.lower():
            test_type = 'RG'
            #sub_name += num
            #sub_name += '_RG'
            #lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4', 'lane_5', 'lane_6']
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4']
        elif 'tandemgang' in files.lower():
            test_type = 'TG'
            #sub_name += num
            #sub_name += '_TG'
            lanes = ['lane_1', 'lane_2', 'lane_3', 'lane_4']

        timeframes = data[:,1] #fractions of a second?
        time = timeframes / 60 #seconds or minutes?
        
        '''OLD VALUES
        COM_x = data [:,14] #sagittal / anterior posterior
        COM_y = data[:,15] # right left
        COM_z = data[:,16] # vertical / up down
        LeftFoot_x = data [:,17]
        LeftFoot_y = data [:,18]
        LeftFoot_z = data [:,19]
        RightFoot_x = data [:,20] #front-back in m
        RightFoot_y = data [:,21] #left-right
        RightFoot_z = data [:,22] #height
        '''

        COM_x = data [:,2] #sagittal / anterior posterior
        COM_y = data[:,3] # right left
        COM_z = data[:,4] # vertical / up down
        LeftFoot_x = data [:,11]
        LeftFoot_y = data [:,12]
        LeftFoot_z = data [:,13]
        RightFoot_x = data [:,14] #front-back in m
        RightFoot_y = data [:,15] #left-right
        RightFoot_z = data [:,16] #height

        #print(f'Test Type: {test_type}')
        
        '''
        print("---- Walk plot ----")
        plt.axis('equal')
        plt.plot(LeftFoot_x, LeftFoot_y)
        plt.axis('equal')
        plt.plot(RightFoot_x, RightFoot_y)
        plt.axis('equal')
        plt.plot(COM_x, COM_y)
        plt.show()
        #plt.plot(time, LeftFoot_y, time, COM_y, time, RightFoot_y)
        '''

        COM_z_time = np.array([time]+[COM_z]) #COM_z values with corresponding time
        COM_x_time = np.array([time]+[COM_x])
        COM_y_time = np.array([time]+[COM_y])

        
        ## Calculate Time at Gait Plate
        maxCOMx = max(COM_x)-1
        minCOMx = min(COM_x)+1


        COM_x_new = copy.copy(COM_x)
        COM_x_new[COM_x_new > maxCOMx] = 9.999
        COM_x_new[COM_x_new < minCOMx] = 9.999
        COM_x_new_time = np.array([time] + [COM_x_new])
        
        #PRINT CENTER OF MASS GRAPHS
        #print("---- lane / time - Plot ----")
        #plt.plot(COM_x_new_time[0,:], COM_x_new_time[1,:])
        #plt.show()

        '''
        frames = pd.read_csv('../data/frames.csv', sep = ';')        
        lane_frames = []

        for lane in range(len(lanes)):
            if int(subject) not in list(frames['subject']):
                print('Subject ist not in frames.csv! Please add frames to the file.')
                break
            lane_start_end = []
            lane_start_end.append(frames['start'][(frames['test'] == test_type) & (frames['lane'] == lane+1) & (frames['subject'] == int(subject))].item())
            lane_start_end.append(frames['end'][(frames['test'] == test_type) & (frames['lane'] == lane+1) & (frames['subject'] == int(subject))].item())
            lane_frames.append(lane_start_end)
        '''
        _, lane_frames = get_frames(COM_x_new_time, sub_name, test_type)
        
        #print("lane_frames:", lane_frames)
        ### Define x,y,z for COM, LF and RF Data per Foot

        # exemplary dictionary:
        # com_data = {'lane_1': [[x_data][y_data][z_data]],
        #             'lane_2': [[x_data][y_data][z_data]]}

         
        com_data = {}
        lf_data = {}
        rf_data = {} 

        ## Build COM data dict per lane
        for lane in range(len(lanes)):
            lane_name = "lane_" + str(lane+1)
            x_data = COM_x_new_time[:, lane_frames[lane][0]:lane_frames[lane][1]]
            y_data = COM_y_time[:, lane_frames[lane][0]:lane_frames[lane][1]]
            z_data = COM_z_time[:, lane_frames[lane][0]:lane_frames[lane][1]]
            com_data[lane_name] = [x_data, y_data, z_data]


        ## Build Left/Right Foot data dict per lane
        for lane in range(len(lanes)):
            lane_name = "lane_" + str(lane+1)
            x_data = LeftFoot_x[lane_frames[lane][0]:lane_frames[lane][1]]
            y_data = LeftFoot_y[lane_frames[lane][0]:lane_frames[lane][1]]
            z_data = LeftFoot_z[lane_frames[lane][0]:lane_frames[lane][1]]
            lf_data[lane_name] = [x_data, y_data, z_data]

            x_data = RightFoot_x[lane_frames[lane][0]:lane_frames[lane][1]]
            y_data = RightFoot_y[lane_frames[lane][0]:lane_frames[lane][1]]
            z_data = RightFoot_z[lane_frames[lane][0]:lane_frames[lane][1]]
            rf_data[lane_name] = [x_data, y_data, z_data]

        
        var_all = []
        v_all = []

        Stride_length_all = [] #actually stride lengths
        Stride_length_all_L = [] 
        Stride_length_all_R = [] 

        step_width_all = []
        LF_height_sum = []
        RF_height_sum = []
        step_lengths = [] # step_lengths

        #swing & stances
        all_stances_L = []
        all_stances_R = []
        all_swings_L = []
        all_swings_R = []

        #percent swing and stances, currently calculated with all
        all_pc_st = []
        all_pc_sw = []
        
        #double support times for each lane in file
        dst_list = []

        for lane in lanes:
            #print(f'-------------------------{lane}-------------------------------')
            COM_xy_Lane_tp_new,  LF_xy_Lanetp_new, RF_xy_Lanetp_new = get_gait(lane, com_data, lf_data, rf_data, plot='off')

            var_lane = statistics.variance(COM_xy_Lane_tp_new[0,:])
            var_all.append(var_lane)
            
            v_lane = abs((COM_xy_Lane_tp_new[1,-1]-COM_xy_Lane_tp_new[1,0])/(com_data[lane][0][0,-1]-com_data[lane][0][0,0])) #(pos_end - pos_start) / (time_end - time_start)
            
            print("1, -1", COM_xy_Lane_tp_new[1,-1], len(COM_xy_Lane_tp_new[1, ]))
            print("0, 0 -1", com_data[lane][0][0,-1], len(com_data[lane][0][0,]))
            
            v_all.append(v_lane) #average of this is the speed
            
            peaks_COM, peaks_left, peaks_right, peaks_left_corrected, peaks_right_corrected, peaks_timeonly, peaks_heightonly, minima_RF, minima_LF = get_peaks(lane, com_data, lf_data, rf_data, plot = 'off') ##
            
            #STRIDE LENGTH
            stride_length, stride_length_L, stride_length_R = calculate_step_stride_length(lane, com_data, lf_data, rf_data)
            Stride_length_all += list(stride_length) #check this metric --> ACTUALLY STRIDE
            Stride_length_all_L += list(stride_length_L)
            Stride_length_all_R += list(stride_length_R)
            
            #STEP LENGTH
            step_lengths += list(get_step_lengths(minima_LF, minima_RF, LF_xy_Lanetp_new, RF_xy_Lanetp_new, lane)) #--> STEPS
            
            #STEP WIDTH
            step_width_all.append(calc_step_width(lane, com_data, lf_data, rf_data))

            LF_height_sum += list(peaks_left_corrected)
            RF_height_sum += list(peaks_right_corrected)
            
            #SWING AND STANCE TIME
            R, L, _, _ = intersect(lane, rf_data, lf_data, minima_RF, minima_LF, plot="off") ##
            swing_R, stance_R, swing_L, stance_L, dst_base = calculate_stance_swing(lane, L, R, lf_data, rf_data, com_data)
            dst = double_support_time(dst_base)
            pc_sw, pc_st = calculate_percent_cycle(swing_R, stance_R, swing_L, stance_L)
            
            dst_list.append(dst)
            
            all_stances_L += list(stance_L)
            all_stances_R += list(stance_R)
            all_swings_L += list(swing_L)
            all_swings_R += list(swing_R)
            
            all_pc_st.append(pc_st)
            all_pc_sw.append(pc_sw)
            
            plt.close()
            
        ## STANCE/SWING ##

        all_stances_L = np.array(all_stances_L).flatten() #flatten to 1d
        all_stances_R = np.array(all_stances_R).flatten()
        all_swings_L = np.array(all_swings_L).flatten()
        all_swings_R = np.array(all_swings_R).flatten()

        # average swing & stance, each leg (4)
        stance_L_avg = np.mean(all_stances_L)
        stance_R_avg = np.mean(all_stances_R)
        swing_L_avg = np.mean(all_swings_L)
        swing_R_avg = np.mean(all_swings_R)

        # average % swing and % stance 
        pc_st_avg = np.mean(all_pc_st)
        pc_sw_avg = np.mean(all_pc_sw)

        # TOTAL stance and swing percentages --> effectively the same
        total = np.sum(all_stances_L) + np.sum(all_stances_R) + np.sum(all_swings_L) + np.sum(all_swings_R)
        total_pc_sw = (np.sum(all_swings_L) + np.sum(all_swings_R)) / total
        total_pc_st = (np.sum(all_stances_L) + np.sum(all_stances_R)) / total

        # variability, both legs (1)
        stances_L_variability = variability(all_stances_L)
        stances_R_variability = variability(all_stances_R)
        swings_L_variability = variability(all_swings_L)
        swings_R_variability = variability(all_swings_R)

        # asymmetry, between legs (1)
        stances_asymmetry = asymmetry(all_stances_L, all_stances_R)
        swings_asymmetry = asymmetry(all_swings_L, all_swings_R)
        
        #double support time
        dst_avg = np.mean(dst_list)

        #interstep & frequency
        InterStepTime_avg, InterStepTime_arr = calc_interstep_all(lanes, com_data, lf_data, rf_data)
        Variability_InterStepTime = variability(InterStepTime_arr)
        Frequency = 1/InterStepTime_avg

        #### Calculate Variance of COMy and Velocity/Speed

        ## Durchschnittsgeschwindigkeit:
        variance_COMy = np.mean(var_all)
        speed = np.mean(v_all)

        #STRIDE
        stride_length_avg = np.mean(Stride_length_all) # Average STRIDE Length, all step lengths from right and left foot...
        stride_length_var = statistics.variance(Stride_length_all) # Variance
        stride_length_variability = variability(Stride_length_all)
        stride_length_asymmetry = asymmetry(Stride_length_all_L, Stride_length_all_R)

        #STEP -- NEW
        ac_step_length_avg = np.mean(step_lengths)
        ac_step_length_variance = statistics.variance(step_lengths)
        ac_step_length_variability = variability(step_lengths)

        step_width_all = [item for sublist in step_width_all for item in sublist] # macht aus einem nested array ein 1D-array
        #print(step_width_all)
        ##plt.plot(step_width_all, 'x')
        step_width_avg = np.mean(step_width_all)
        var_step_width = statistics.variance(step_width_all)
        step_width_variability = variability(step_width_all)

        #### Calculate Foot Height at Maximum Elevation
        LF_height_avg = np.mean(LF_height_sum)
        RF_height_avg = np.mean(RF_height_sum)

        foot_height_asymmetry = asymmetry(LF_height_sum, RF_height_sum)
        LF_height_variability = variability(LF_height_sum)
        RF_height_variability = variability(RF_height_sum)
        
        df = pd.DataFrame([trial_num,
                    test_type, 
                   stance_L_avg,
                   swing_L_avg,
                   stance_R_avg,
                   swing_R_avg,
                   stances_L_variability,
                   swings_L_variability,
                   stances_R_variability,
                   swings_R_variability,
                   stances_asymmetry,
                   swings_asymmetry,
                   total_pc_sw,
                   total_pc_st,
                   dst_avg,
                   InterStepTime_avg,
                   Variability_InterStepTime,
                   Frequency,
                   variance_COMy,
                   speed,
                   stride_length_avg,
                   stride_length_var,
                   stride_length_variability,
                   stride_length_asymmetry,
                   ac_step_length_avg,
                   ac_step_length_variance,
                   ac_step_length_variability,
                   step_width_avg,
                   var_step_width,
                   step_width_variability,
                   LF_height_avg,
                   RF_height_avg,
                   LF_height_variability,
                   RF_height_variability,
                   foot_height_asymmetry], index= INDICES, columns = [sub_name])

        all_results = pd.concat([all_results, df.T], axis = 0)
        all_results.to_csv('all_pat_042926.csv')

        #df.to_csv(f'../export/{sub_name}.csv')
        #df.to_csv(f'out/{sub_name}.csv') 
        #print(f'Analysis done for subject: {sub_name}')
        #plt.close()
        counter += 1
        
    except Exception as e:
        print("file_strip", file_strip)
        print("sub_name", sub_name)
        print("trial_num", trial_num)
        
        print(f'Starting analysis for subject: {subject}')
        print(f"---------index {counter}---------")
        print(f'Exception occured: {e}')
        print(traceback.format_exc())
        error_subj.append(str(sub_name))
        error_trace.append(traceback.format_exc())
        error_idx.append(counter)
        counter += 1
        
        #printing plots only for debugging
        print("trying debugging plots")
        try:
            for lane in lanes:
                _ = get_peaks(lane, com_data, lf_data, rf_data, plot = 'off') ##
                _ = intersect(lane, rf_data, lf_data, minima_RF, minima_LF, plot = 'off') ##
        except Exception as e2:
            print(f'Exception occured: {e}')
            print(traceback.format_exc())

       
       
        

#print(results)
#results.to_csv(f'../export/results_walks.csv')
#results.to_excel(f'../export/results_walks.xlsx')


1, -1 3.714684207630067 269
0, 0 -1 13.066666666666666 269
1, -1 1.020829962115548 197
0, 0 -1 20.933333333333334 197
1, -1 3.714783601817014 200
0, 0 -1 28.183333333333334 200
1, -1 1.0310841611706143 213
0, 0 -1 36.03333333333333 213
1, -1 4.0235816367322785 396
0, 0 -1 12.7 396
1, -1 0.8404339523117468 300
0, 0 -1 23.05 300
1, -1 4.074456708559329 263
0, 0 -1 34.71666666666667 263
1, -1 0.8301508103558592 260
0, 0 -1 46.5 260
1, -1 4.331863454495335 315
0, 0 -1 18.8 315
1, -1 1.0620608227580186 282
0, 0 -1 31.333333333333332 282
1, -1 4.350482364880957 282
0, 0 -1 42.266666666666666 282
1, -1 1.0639535387910208 293
0, 0 -1 51.6 293
1, -1 4.332363689866361 279
0, 0 -1 61.93333333333333 279
1, -1 1.062076104284715 285
0, 0 -1 71.75 285
file_strip 132-003-tandemgang_Seg_RightFoot_LeftFoot_RightForeArm_LeftForeArm_Pelvis_Data_position_acceleration.txt
sub_name 132
trial_num 1
Starting analysis for subject: 132
---------index 3---------
Exception occured: index 2 is out of bounds for axi

/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


1, -1 0.8680950616416626 196
0, 0 -1 45.666666666666664 196
1, -1 3.4814934478747115 292
0, 0 -1 15.566666666666666 292
file_strip 101-2-001-NormalerGang_Seg_RightFoot_LeftFoot_RightForeArm_LeftForeArm_Pelvis_Neck_Data_position_acceleration.txt
sub_name 101
trial_num 2
Starting analysis for subject: 101
---------index 10---------
Exception occured: index 0 is out of bounds for axis 0 with size 0
Traceback (most recent call last):
  File "/var/folders/nm/1x6qm5g522n2n1mmwkfs5m8h0000gn/T/ipykernel_49509/3417870803.py", line 216, in <module>
    step_lengths += list(get_step_lengths(minima_LF, minima_RF, LF_xy_Lanetp_new, RF_xy_Lanetp_new, lane)) #--> STEPS
                         ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/viggy/Desktop/INM/gait_analysis/func_stepFrequency.py", line 458, in get_step_lengths
    if np.abs(left_steps[0] - center) > np.abs(right_steps[0] - center): #left first, add first two steps
              ~~~~~~~~~~

/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


1, -1 1.008457518810153 428
0, 0 -1 33.833333333333336 428
1, -1 3.880489430877352 596
0, 0 -1 50.833333333333336 596
1, -1 1.00519294294638 618
0, 0 -1 67.28333333333333 618
double peak detected at end of signal, moving on...
1, -1 4.326528647945975 654
0, 0 -1 17.183333333333334 654
1, -1 0.6381495757618513 253
0, 0 -1 24.833333333333332 253
1, -1 4.3097130918473665 245
0, 0 -1 33.15 245
1, -1 0.6511818559468965 257
0, 0 -1 41.06666666666667 257
1, -1 4.318867898088291 257
0, 0 -1 48.38333333333333 257
1, -1 0.6557064353814781 255
0, 0 -1 56.28333333333333 255
1, -1 4.249458245991854 294
0, 0 -1 14.916666666666666 294
1, -1 0.8948757487968011 256
0, 0 -1 23.05 256
1, -1 4.242916057598988 285
0, 0 -1 32.61666666666667 285
1, -1 0.8924863521931332 190
0, 0 -1 42.06666666666667 190
1, -1 4.25608815739934 216
0, 0 -1 49.666666666666664 216
1, -1 0.8968984816671924 234
0, 0 -1 58.166666666666664 234
1, -1 4.19862927076719 739
0, 0 -1 15.033333333333333 739
1, -1 0.6110288196445274 389
0, 

...

/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


1, -1 3.9233100008461723 191
0, 0 -1 36.083333333333336 191
1, -1 0.7100883296584547 194
0, 0 -1 42.56666666666667 194
1, -1 3.9434719196098245 267
0, 0 -1 17.466666666666665 267
1, -1 0.8833916718099972 230
0, 0 -1 25.383333333333333 230
1, -1 3.9389742228746023 213
0, 0 -1 32.35 213
1, -1 0.8959767320337946 225
0, 0 -1 39.583333333333336 225
1, -1 3.944460543603375 204
0, 0 -1 45.983333333333334 204
1, -1 0.8977160233172182 217
0, 0 -1 53.016666666666666 217
1, -1 4.0196779038780415 223
0, 0 -1 13.4 223
1, -1 0.6486820360093292 180
0, 0 -1 19.816666666666666 180
1, -1 4.0360179474767675 189
0, 0 -1 26.95 189
1, -1 0.6762222818386092 176
0, 0 -1 33.733333333333334 176
1, -1 4.064018206199551 183
0, 0 -1 40.05 183
1, -1 0.7096688897225505 184
0, 0 -1 46.81666666666667 184
1, -1 3.978451653321242 284
0, 0 -1 12.5 284
1, -1 0.6647211840104472 258
0, 0 -1 22.566666666666666 258
1, -1 3.986229411774158 305
0, 0 -1 32.85 305
1, -1 0.6700382960602397 254
0, 0 -1 42.68333333333333 254
file_st

/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


1, -1 0.9997336338872169 1064
0, 0 -1 164.7 1064
1, -1 3.530262447770194 1057
0, 0 -1 36.166666666666664 1057
double peak detected at end of signal, moving on...
1, -1 0.7861496671353505 824
0, 0 -1 70.75 824
1, -1 3.5117812795416117 723
0, 0 -1 98.58333333333333 723
1, -1 0.7911659325514206 804
0, 0 -1 127.73333333333333 804
1, -1 4.18720518250493 835
0, 0 -1 17.316666666666666 835
1, -1 0.6012355600463217 453
0, 0 -1 32.7 453
1, -1 4.171949793462846 451
0, 0 -1 47.2 451
1, -1 0.6017094799372882 478
0, 0 -1 63.733333333333334 478
1, -1 4.152844449565596 486
0, 0 -1 79.2 486
1, -1 0.6037933288554349 437
0, 0 -1 95.28333333333333 437
1, -1 4.023961137188461 388
0, 0 -1 22.1 388
1, -1 0.8196584218236521 335
0, 0 -1 33.55 335
1, -1 4.020459915010264 306
0, 0 -1 45.38333333333333 306
1, -1 0.8228486684081038 296
0, 0 -1 55.516666666666666 296
1, -1 4.004330156262311 320
0, 0 -1 66.86666666666666 320
1, -1 0.8320205833492688 274
0, 0 -1 78.4 274
1, -1 3.922735499495194 321
0, 0 -1 13.5 321


/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [9]:
for i in all_results["double_support_time"]:
    print(i)

0.0404021644566128


In [5]:
all_results.to_csv('out/all_results_dst.csv')


In [8]:
len(error_subj)

28

In [6]:
demos = pd.read_csv("Demographics_PD.csv")

In [8]:
#subset to get rid of
demos_sub = demos.iloc[0:28, ]

demos_sub["ID"] = demos_sub["ID"].astype(int)
demos_sub["ID"] = demos_sub["ID"].astype(str)

#replace NV values with as control
demos_sub["Control"][14:16] = pd.Series(["1","1"])

#create mapping from control to subject ID
map = demos_sub[["ID", "Control"]]
map.set_index('ID', inplace=True)

mapping = map["Control"].to_dict()


demos_sub["Control"] = demos_sub["ID"].map(mapping)
all_results["ID"] = all_results.index
all_results["Control"] = all_results["ID"].map(mapping) #create Control column

/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demos_sub["ID"] = demos_sub["ID"].astype(int)
/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demos_sub["ID"] = demos_sub["ID"].astype(str)
/var/folders/sp/gln8rmf11713l0sb7h79sbf00000gn/T/ipykernel_58496/1655282577.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a D

In [9]:
all_results.to_csv("out/all_results_dst_ctrl.csv")